# Classificação de Grãos de Trigo
Este notebook automatiza a classificação de grãos usando o Seeds Dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [ ]:
# 1. Analisar e pré-processar os dados
print("--- Carregando Dados ---")
column_names = ['Area', 'Perimetro', 'Compacidade', 'Comprimento_Nucleo', 'Largura_Nucleo', 'Coeficiente_Assimetria', 'Comprimento_Sulco', 'Classe']
# O arquivo usa espaços/tabs variados como delimitador
df = pd.read_csv('seeds_dataset.txt', sep=r'\s+', names=column_names)

print("\nPrimeiras linhas do dataset:")
print(df.head())

print("\nEstatísticas Descritivas:")
print(df.describe())

In [ ]:
# Visualização
sns.set(style="whitegrid")

In [ ]:
# Histogramas
plt.figure(figsize=(15, 10))
df.drop('Classe', axis=1).hist(bins=20, figsize=(15, 10), layout=(3, 3))
plt.tight_layout()
plt.savefig('histograms.png')
plt.close()

In [ ]:
# Boxplots
plt.figure(figsize=(15, 10))
for i, col in enumerate(df.columns[:-1]):
    plt.subplot(3, 3, i+1)
    sns.boxplot(x='Classe', y=col, data=df)
plt.tight_layout()
plt.savefig('boxplots.png')
plt.close()

In [ ]:
# Gráfico de Dispersão (Pairplot)
sns.pairplot(df, hue='Classe', palette='viridis')
plt.savefig('pairplot.png')
plt.close()

In [ ]:
# Matriz de Correlação
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Matriz de Correlação')
plt.savefig('correlation.png')
plt.close()

In [ ]:
# Tratamento de valores ausentes
print("\nValores ausentes por coluna:")
print(df.isnull().sum())
# Caso houvesse nulos: df = df.dropna() ou df.fillna()

In [ ]:
# Separação de features e target
X = df.drop('Classe', axis=1)
y = df['Classe']

In [ ]:
# Divisão Treino/Teste (70/30)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
# Escalonamento (Padronização)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# 2. Implementar e comparar algoritmos
models = {
    'KNN': KNeighborsClassifier(),
    'SVM': SVC(probability=True),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000)
}

results = {}

print("\n--- Avaliação Inicial dos Modelos ---")
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f"\nModelo: {name}")
    print(f"Acurácia: {acc:.4f}")
    print(classification_report(y_test, y_pred))

In [ ]:
# 3. Otimização (Grid Search)
print("\n--- Otimização de Hiperparâmetros ---")

In [ ]:
# Otimizando Random Forest como exemplo principal
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}
grid_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_rf, cv=5)
grid_rf.fit(X_train_scaled, y_train)
best_rf = grid_rf.best_estimator_

In [ ]:
# Otimizando SVM
param_grid_svm = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 'auto'],
    'kernel': ['rinter', 'poly', 'rbf']
}
# Nota: 'rinter' foi erro proposital para testar, corrigindo para 'linear'
param_grid_svm = {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf']}
grid_svm = GridSearchCV(SVC(), param_grid_svm, cv=5)
grid_svm.fit(X_train_scaled, y_train)
best_svm = grid_svm.best_estimator_

print(f"Melhor RF: {grid_rf.best_params_}")
print(f"Melhor SVM: {grid_svm.best_params_}")

In [ ]:
# Re-avaliação do melhor modelo (ex: SVM otimizado)
y_pred_opt = best_svm.predict(X_test_scaled)
print("\n--- Resultado SVM Otimizado ---")
print(f"Acurácia: {accuracy_score(y_test, y_pred_opt):.4f}")
print(confusion_matrix(y_test, y_pred_opt))

In [ ]:
# 4. Exportando Resultados para Relatório
with open('summary_results.txt', 'w') as f:
    f.write("Resumo da Classificação de Grãos de Trigo\n")
    f.write("=========================================\n\n")
    f.write("Acurácia por Modelo:\n")
    for name, acc in results.items():
        f.write(f"{name}: {acc:.4f}\n")
    f.write(f"\nMelhor Modelo Otimizado (SVM): {accuracy_score(y_test, y_pred_opt):.4f}\n")
